**Data Info**

train.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용
* first_party_winner : 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

test.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용

sample_submission.csv - 제출 양식
* ID : 사건 샘플 ID
* first_party_winner : 예측한 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

# Fine-tune Model: RoBERTa

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

In [2]:
MODEL_NAME = "roberta-base"   # 데이터가 작으니 large보다 base로 먼저 확인 후 스케일업 추천
MAX_LEN = 384
N_FOLDS = 5
SEED = 42

In [4]:
train = pd.read_csv("./train.csv")
test = pd.read_csv("./test.csv")

In [5]:
def build_text(row):
    return f"{row['first_party']} vs {row['second_party']}: {row['facts']}"

train["text"] = train.apply(build_text, axis=1)
test["text"] = test.apply(build_text, axis=1)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
class JudgmentDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

X = train["text"].values
y = train["first_party_winner"].values
X_test = test["text"].values

In [8]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_pred = np.zeros(len(train))
test_probs_sum = np.zeros((len(test), 2))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== Fold {fold+1}/{N_FOLDS} =====")

    train_ds = JudgmentDataset(X[tr_idx], y[tr_idx])
    valid_ds = JudgmentDataset(X[va_idx], y[va_idx])
    test_ds = JudgmentDataset(X_test)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./ckpt_fold{fold}",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        bf16=True,

        logging_steps=20,
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    va_logits = trainer.predict(valid_ds).predictions
    oof_pred[va_idx] = np.argmax(va_logits, axis=1)

    test_logits = trainer.predict(test_ds).predictions
    test_probs_sum += torch.softmax(torch.tensor(test_logits), dim=1).numpy()

    acc = accuracy_score(y[va_idx], oof_pred[va_idx])
    f1 = f1_score(y[va_idx], oof_pred[va_idx], average="macro")
    print(f"Fold {fold+1} | Acc: {acc:.4f} | Macro F1: {f1:.4f}")

    del model, trainer
    torch.cuda.empty_cache()

print("\n=== Overall OOF ===")
print("Accuracy:", accuracy_score(y, oof_pred))
print("Macro F1:", f1_score(y, oof_pred, average="macro"))


===== Fold 1/5 =====


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 5 | Acc: 0.6646 | Macro F1: 0.3993

=== Overall OOF ===
Accuracy: 0.6674737691686844
Macro F1: 0.43870073891625616


In [9]:
test_pred = np.argmax(test_probs_sum, axis=1)
submit = pd.read_csv("./sample_submission.csv")
submit["first_party_winner"] = test_pred
submit.to_csv("./roberta_submit.csv", index=False)
print("Done")

Done


In [ ]:
# 나중에 DeBERTa와 앙상블할 때 쓸 수 있도록 확률(softmax) 저장
np.save("roberta_test_probs.npy", test_probs_sum / N_FOLDS)
np.save("roberta_oof_pred.npy", oof_pred)
print("Done")